# 📌 Tarea 3 (Módulo 3) — Notebook resuelto

Este notebook **viene del curso** (Stock Markets Analytics Zoomcamp 2026, DataTalksClub). Todo el código original es del curso.
Lo único agregado son las celdas marcadas con **✅ SOLUCIÓN**, que responden las preguntas de la tarea. Cada una tiene arriba una explicación en lenguaje sencillo.

**Cómo correrlo:** en Colab, *Entorno de ejecución → Ejecutar todas*. Tarda unos 15–25 minutos (el paso más lento es entrenar los árboles de la Pregunta 4).
Al final hay una tabla **📝 Resumen de respuestas** con las 4 respuestas numéricas.

| Pregunta | Dónde está la solución |
|---|---|
| Q1 | Sección 0.2 (crear `month_wom`) y final de la 0.3 (correlación) |
| Q2 | Sección 1.2 (reglas `pred3`, `pred4`) y justo después del cálculo de precisión |
| Q3, Q4, Q5 | Al final del notebook |

In [1]:
!pip install yfinance

In [ ]:
# read files shared via google-drive-link
# https://stackoverflow.com/questions/62759748/downloading-data-from-a-shared-google-drive-link-in-google-colab

!pip uninstall gdown -y && pip install gdown
!gdown -V

Found existing installation: gdown 5.2.2
Uninstalling gdown-5.2.2:
  Successfully uninstalled gdown-5.2.2


In [ ]:
# IMPORTS
import numpy as np
import pandas as pd

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.graph_objects as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt

# 0) Dataset for Modeling: Final Preparations

## 0.1) Importing data from Drive & defining variable sets
* automated version need to have a daily updated file/database entries

In [ ]:
import gdown

# https://stackoverflow.com/questions/62759748/downloading-data-from-a-shared-google-drive-link-in-google-colab
# truncated data from Module 2: https://drive.google.com/file/d/1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5/view?usp=sharing
!gdown https://drive.google.com/file/d/1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5/view?usp=sharing -O /content/

In [ ]:
# full dataset for 33 stocks
df_full = pd.read_parquet("/content/stocks_df_combined_2026_09_18.parquet.brotli", )


In [ ]:
df_full.info()

In [ ]:
df_full.keys()

In [ ]:
# growth indicators (but not future growth)
GROWTH = [g for g in df_full.keys() if (g.find('growth_')==0)&(g.find('future')<0)]
GROWTH

In [ ]:
# leaving only Volume ==> generate ln(Volume)
OHLCV = ['Open','High','Low','Close','Adj Close_x','Volume']

In [ ]:
CATEGORICAL = ['Month', 'Weekday', 'Ticker', 'ticker_type']

In [ ]:
TO_PREDICT = [g for g in df_full.keys() if (g.find('future')>=0)]
TO_PREDICT

In [ ]:
TO_DROP = ['Year','Date','index_x', 'index_y', 'index', 'Quarter','Adj Close_y'] + CATEGORICAL + OHLCV
TO_DROP

In [ ]:
# a small hack to avoid "divide by zero encountered in log"
df_full['ln_volume'] = df_full.Volume.replace(0, np.nan).fillna(1e-9).apply(lambda x: np.log(x))

In [ ]:
# manually defined features
CUSTOM_NUMERICAL = ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative','volatility', 'ln_volume']

In [ ]:
# All Supported Ta-lib indicators: https://github.com/TA-Lib/ta-lib-python/blob/master/docs/funcs.md

TECHNICAL_INDICATORS = ['adx', 'adxr', 'apo', 'aroon_1','aroon_2', 'aroonosc',
 'bop', 'cci', 'cmo','dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext',
 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix',
 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo',
 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk',
 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr',
 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase',
 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine',
 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice']

In [ ]:
TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find('cdl')>=0]
print(f'Technical patterns count = {len(TECHNICAL_PATTERNS)}, examples = {TECHNICAL_PATTERNS[0:5]}')


In [ ]:
MACRO = ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS',
 'DGS1', 'DGS5', 'DGS10']

In [ ]:
NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO

In [ ]:
# CHECK: NO OTHER INDICATORS LEFT
OTHER = [k for k in df_full.keys() if k not in OHLCV + CATEGORICAL + NUMERICAL + TO_DROP + TO_PREDICT]
OTHER

In [ ]:
df_full.Ticker.nunique()

In [ ]:
# tickers, min-max date, count of daily observations
df_full.groupby(['Ticker'])['Date'].agg(['min','max','count'])

In [ ]:
# truncated df_full with 25 years of data (and defined growth variables)
df = df_full[df_full.Date>='2000-01-01']
df.info()

In [ ]:
# let's look at the features count and df size:
df[NUMERICAL].info()

## 0.2) [Code snippet 1] Generating dummies

In [ ]:
# what are the categorical features?
CATEGORICAL

In [ ]:
df['Weekday'] = df['Weekday'].astype(str)
df['Month'] = df['Month'].dt.month.astype(str) # Convert to string so get_dummies creates dummy columns properly

## ✅ SOLUCIÓN – Pregunta 1 (parte A): crear la columna `month_wom`
**¿Qué hacemos?** A cada día le ponemos una etiqueta que dice *qué mes es* y *qué semana de ese mes es*. Ejemplo: el 3 de octubre → `October_w1`; el 16 de octubre → `October_w3`.

**¿Cómo?**
- `df['Date'].dt.month_name()` saca el nombre del mes de la fecha (`'October'`).
- `(día - 1) // 7 + 1` calcula la semana: los días 1–7 son la semana 1, los días 8–14 la semana 2, y así. (`//` es división entera: 13 // 7 = 1.)
- Unimos los dos textos con `'_w'` en medio.
- Agregamos `'month_wom'` a la lista `CATEGORICAL`, para que la celda siguiente (`get_dummies`) la convierta en ~60 columnas de 0/1, una por cada combinación mes-semana.

Va **antes** de `get_dummies` para que las nuevas columnas se creen junto con las demás.

In [ ]:
# ✅ SOLUCIÓN — Pregunta 1 (parte A)
semana_del_mes = (df['Date'].dt.day - 1) // 7 + 1            # 1, 2, 3, 4 o 5
df['month_wom'] = df['Date'].dt.month_name() + '_w' + semana_del_mes.astype(str)

# agregar la nueva variable a la lista de categóricas (sin duplicarla si se corre dos veces)
if 'month_wom' not in CATEGORICAL:
    CATEGORICAL.append('month_wom')

print(CATEGORICAL)
print('Valores distintos de month_wom:', df['month_wom'].nunique())
df[['Date', 'month_wom']].head()

In [ ]:
# Generate dummy variables (no need for bool, let's have int32 instead)
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')

In [ ]:
# TODO 1: define more categorical features, e.g. all combinations for <September+weekday>  (you'll see that September is actually an important dummy in one of the models)

In [ ]:
dummy_variables.info()

In [ ]:
# get dummies names in a list
DUMMIES = dummy_variables.keys().to_list()

In [ ]:
# Concatenate the dummy variables with the original DataFrame
df_with_dummies = pd.concat([df, dummy_variables], axis=1)

In [ ]:
df_with_dummies[NUMERICAL+DUMMIES].info()

## 0.3) [Code Snippet 2] Correlation analysis
* first approximation of "important" variables correlated with all variables we want to predict (TO_PREDICT)

In [ ]:
TO_PREDICT

In [ ]:
corr_is_positive_growth_30d_future = df_with_dummies[NUMERICAL+DUMMIES+TO_PREDICT].corr()['is_positive_growth_30d_future']

In [ ]:
# create a dataframe for an easy way to sort
corr_is_positive_growth_30d_future_df = pd.DataFrame(corr_is_positive_growth_30d_future)

In [ ]:
corr_is_positive_growth_30d_future_df.sort_values(by='is_positive_growth_30d_future').head(5)

In [ ]:
corr_is_positive_growth_30d_future_df.sort_values(by='is_positive_growth_30d_future').tail(8)

In [ ]:
corr_growth_future_30d = df_with_dummies[NUMERICAL+DUMMIES+TO_PREDICT].corr()['growth_future_30d']

In [ ]:
corr_growth_future_30d_df = pd.DataFrame(corr_growth_future_30d)

In [ ]:
corr_growth_future_30d_df.sort_values(by='growth_future_30d').head(5)

In [ ]:
corr_growth_future_30d_df.sort_values(by='growth_future_30d').tail(8)

## ✅ SOLUCIÓN – Pregunta 1 (parte B): ¿qué semana del mes se relaciona más con que la acción suba?
La **correlación** mide qué tanto dos cosas se mueven juntas (de -1 a 1). El signo solo dice la dirección; nos interesa la **fuerza**, por eso usamos el **valor absoluto** (`abs`).

La tabla `corr_is_positive_growth_30d_future_df` (calculada unas celdas arriba) ya incluye las nuevas columnas. Solo tenemos que:
1. quedarnos con las filas cuyo nombre empieza con `month_wom_`,
2. crear `abs_corr` = valor absoluto,
3. ordenar de mayor a menor y tomar la primera, redondeada a 3 decimales.

In [ ]:
# ✅ SOLUCIÓN — Pregunta 1 (parte B)
print('Total de dummies:', len(DUMMIES), '| de month_wom:', len([d for d in DUMMIES if d.startswith('month_wom_')]))

corr_wom = corr_is_positive_growth_30d_future_df[
    corr_is_positive_growth_30d_future_df.index.str.startswith('month_wom_')
].copy()
corr_wom['abs_corr'] = corr_wom['is_positive_growth_30d_future'].abs()
corr_wom = corr_wom.sort_values(by='abs_corr', ascending=False)

respuesta_q1 = round(corr_wom['abs_corr'].iloc[0], 3)
print(f'RESPUESTA Q1: {respuesta_q1}  (variable: {corr_wom.index[0]})')
corr_wom.head(10)

## 0.4) [Code snippet 3] Temporal split of ~25 years of data (by date)

In [ ]:
def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15, test_prop=0.15):
    """
    Splits a DataFrame into three buckets based on the temporal order of the 'Date' column.

    Args:
        df (DataFrame): The DataFrame to split.
        min_date (str or Timestamp): Minimum date in the DataFrame.
        max_date (str or Timestamp): Maximum date in the DataFrame.
        train_prop (float): Proportion of data for training set (default: 0.6).
        val_prop (float): Proportion of data for validation set (default: 0.2).
        test_prop (float): Proportion of data for test set (default: 0.2).

    Returns:
        DataFrame: The input DataFrame with a new column 'split' indicating the split for each row.
    """
    # Define the date intervals
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)

    # Assign split labels based on date ranges
    split_labels = []
    for date in df['Date']:
        if date <= train_end:
            split_labels.append('train')
        elif date <= val_end:
            split_labels.append('validation')
        else:
            split_labels.append('test')

    # Add 'split' column to the DataFrame
    df['split'] = split_labels

    return df

In [ ]:
min_date_df = df_with_dummies.Date.min()
max_date_df = df_with_dummies.Date.max()

df_with_dummies = temporal_split(df_with_dummies,
                                 min_date = min_date_df,
                                 max_date = max_date_df)

In [ ]:
df_with_dummies['split'].value_counts()/len(df_with_dummies)

In [ ]:
# remove the "segmentation" problem (warning message on df performance after many joins and data transformations)
new_df = df_with_dummies.copy()

# 1) Modeling: "rule of thumb" or hand-predictions

In [ ]:
new_df.Ticker.value_counts()

In [ ]:
# let's have a time-series or one ticker
df_amzn = new_df[new_df.Ticker=='AMZN']

In [ ]:
fig = px.line(df_amzn, x="Date", y="Close_x", title='AMZN daily close price')
fig.show()

In [ ]:
# TRAIN / VALIDATION/ TEST split

# Calculate the lengths of each split -- this is not temporal split, but based on the number of observations --> "classic" split with no shuffle
total_length = len(df_amzn)
train_length = int(0.8 * total_length)
val_length = int(0.1 * total_length)

# Split the data
train_data = df_amzn.iloc[:train_length]
val_data = df_amzn.iloc[train_length:train_length+val_length]
test_data = df_amzn.iloc[train_length+val_length:]

# Plot the data
fig = px.line(title='AMZN Close price daily for three time intervals Train/Validation/Test')
fig.add_scatter(x=train_data['Date'], y=train_data['Close_x'], mode='lines', name='Train', line=dict(color='blue'))
fig.add_scatter(x=val_data['Date'], y=val_data['Close_x'], mode='lines', name='Validation', line=dict(color='orange'))
fig.add_scatter(x=test_data['Date'], y=test_data['Close_x'], mode='lines', name='Test', line=dict(color='green'))

# Update layout to center the title
fig.update_layout(title=dict(x=0.5))

fig.show()

In [ ]:
# HIST ON NVDA DATA Train/Test/Validation
import plotly.graph_objects as go

# Plot the data
fig = go.Figure()

# Add histograms for each split
fig.add_trace(go.Histogram(x=train_data['growth_future_30d'], name='Train', opacity=0.75))
fig.add_trace(go.Histogram(x=val_data['growth_future_30d'], name='Validation', opacity=0.75))
fig.add_trace(go.Histogram(x=test_data['growth_future_30d'], name='Test', opacity=0.75))

fig.update_layout(barmode='overlay', title='Distribution of growth_future_30d for AMAZON (AMZN) Growth by Train/Valid/Test sets')
fig.update_traces(opacity=0.75)

fig.show()


In [ ]:
# HIST ON ALL DATA Train/Test/Validation
# comment: there are some outliers and hard co compare the distributions
import plotly.graph_objects as go

# Plot the data
fig = go.Figure()

# Add histograms for each split
fig.add_trace(go.Histogram(x=df_with_dummies[df_with_dummies.split=='train']['growth_future_30d'], name='Train', opacity=0.75))
fig.add_trace(go.Histogram(x=df_with_dummies[df_with_dummies.split=='validation']['growth_future_30d'], name='Validation', opacity=0.75))
fig.add_trace(go.Histogram(x=df_with_dummies[df_with_dummies.split=='test']['growth_future_30d'], name='Test', opacity=0.75))

fig.update_layout(barmode='overlay', title='Distribution of growth_future_30d for All tickers Growth by Train/Valid/Test sets')
fig.update_traces(opacity=0.75)

fig.show()

In [ ]:
# Assuming df_with_dummies is your DataFrame containing the 'growth_future_5d' variable and 'split' column

# Create the histogram
fig = px.histogram(new_df,
                   x="growth_future_30d",
                   color="split",
                   marginal="box", # or violin, rug
                   )
                  #  hover_data=new_df.growth_future_5d)

# Show the histogram
fig.show()

In [ ]:
new_df.groupby(by='split')['growth_future_30d'].describe()

In [ ]:
# is_positive_growth_5d_future: ONLY AMZN stock

# Count occurrences of 0 and 1 for each split
train_counts = train_data['is_positive_growth_30d_future'].value_counts(normalize=True)*100
val_counts = val_data['is_positive_growth_30d_future'].value_counts(normalize=True)*100
test_counts = test_data['is_positive_growth_30d_future'].value_counts(normalize=True)*100

# Plot the data
fig = go.Figure()

# # Add stacked bar charts for each split
# fig.add_trace(go.Bar(x=['Train', 'Validation', 'Test'], y=[train_counts[0], val_counts[0], test_counts[0]], name='0'))
# fig.add_trace(go.Bar(x=['Train', 'Validation', 'Test'], y=[train_counts[1], val_counts[1], test_counts[1]], name='1'))


# Add stacked bar charts for each split
fig.add_trace(go.Bar(x=['Train', 'Validation', 'Test'], y=[train_counts[0], val_counts[0], test_counts[0]], name='0',
                     text=[f'{count:.2f}%' for count in [train_counts[0], val_counts[0], test_counts[0]]], textposition='auto'))
fig.add_trace(go.Bar(x=['Train', 'Validation', 'Test'], y=[train_counts[1], val_counts[1], test_counts[1]], name='1',
                     text=[f'{count:.2f}%' for count in [train_counts[1], val_counts[1], test_counts[1]]], textposition='auto'))


fig.update_layout(barmode='stack', title='Distribution of is_positive_growth_30d_future by Train/Validation/Test for AMZN stock')
fig.update_traces(opacity=0.75)

fig.show()

In [ ]:
# ALL STOCKS: train/test/validation is more similar

# Count occurrences of 0 and 1 for each split
train_counts = df_with_dummies[df_with_dummies.split=='train']['is_positive_growth_30d_future'].value_counts(normalize=True)*100
val_counts = df_with_dummies[df_with_dummies.split=='validation']['is_positive_growth_30d_future'].value_counts(normalize=True)*100
test_counts = df_with_dummies[df_with_dummies.split=='test']['is_positive_growth_30d_future'].value_counts(normalize=True)*100

# Plot the data
fig = go.Figure()


# Add stacked bar charts for each split
fig.add_trace(go.Bar(x=['Train', 'Validation', 'Test'], y=[train_counts[0], val_counts[0], test_counts[0]], name='0',
                     text=[f'{count:.2f}%' for count in [train_counts[0], val_counts[0], test_counts[0]]], textposition='auto'))
fig.add_trace(go.Bar(x=['Train', 'Validation', 'Test'], y=[train_counts[1], val_counts[1], test_counts[1]], name='1',
                     text=[f'{count:.2f}%' for count in [train_counts[1], val_counts[1], test_counts[1]]], textposition='auto'))


fig.update_layout(barmode='stack', title='Distribution of is_positive_growth_30d_future by Train/Validation/Test for ALL stocks')
fig.update_traces(opacity=0.75)

fig.show()

# 1) Modeling: "rule of thumb" or hand-predictions

## 1.1) Review all the inputs again

In [ ]:
# remove the "segmentation" problem (warning message on df performance after many joins and data transformations)
new_df = df_with_dummies.copy()

In [ ]:
# Full dataframe (transformed and truncated to 25 years)
new_df.info()

In [ ]:
# check one record: it has abs. values, text, and numbers
new_df.head(1)

In [ ]:
# time split on train/validation/test: FIXED dates of split, approx. 70%, 15%, 15% split
new_df.groupby(['split'])['Date'].agg({'min','max','count'})

In [ ]:
# what we try to predict
new_df[TO_PREDICT].head(1)

In [ ]:
# to be used as features
new_df[NUMERICAL+DUMMIES].head(1)

## 1.2) [Code Snippet 3] Manual "hand rule" predictions
* CCI (binary, on technical indicator CCI)
* growth_1d>1
* growth_1d>1 & growth_snp500_1d>1
*

In [ ]:
# why does it work?
# compare a vector (pandas.core.series.Series) with scalar 200 ==> element_wise comparison with the number ==> generate a new DataFrame
# (only 5 values)
pd.DataFrame(new_df.cci>200).head()

In [ ]:
# generate manual predictions
# Let's label all prediction features with prefix "pred"
new_df['pred0_manual_cci'] = (new_df.cci>200).astype(int)
new_df['pred1_manual_prev_g1'] = (new_df.growth_30d>1).astype(int)
new_df['pred2_manual_prev_g1_and_snp'] = ((new_df['growth_30d'] > 1) & (new_df['growth_snp500_30d'] > 1)).astype(int)

In [ ]:
# TODO 2: find more "hand rules" - can get it from decision trees important factors, or randomly build on other most popular macro/tech indicators/ manual_features

## ✅ SOLUCIÓN – Pregunta 2 (parte A): dos reglas “a mano” nuevas
Una regla a mano es una apuesta simple: *“si se cumple esta condición, digo que la acción va a subir (1); si no, digo 0”*.
Estas dos reglas usan tasas de interés de bonos del gobierno de EE. UU. a 10 y 5 años (`DGS10`, `DGS5`) y la tasa de la Reserva Federal (`FEDFUNDS`):
- **pred3:** bonos a 10 años ≤ 4.5% **y** a 5 años ≤ 4% → “sube”.
- **pred4:** bonos a 10 años > 4% **y** tasa de la Fed ≤ 4.795% → “sube”.

`&` significa “y”; `.astype(int)` convierte Verdadero/Falso en 1/0. Las celdas de abajo (PREDICTIONS, IS_CORRECT) ya las incluyen solas.

In [ ]:
# ✅ SOLUCIÓN — Pregunta 2 (parte A)
new_df['pred3_manual_dgs10_5'] = ((new_df['DGS10'] <= 4.5) & (new_df['DGS5'] <= 4)).astype(int)
new_df['pred4_manual_dgs10_fedfunds'] = ((new_df['DGS10'] > 4) & (new_df['FEDFUNDS'] <= 4.795)).astype(int)

new_df[['DGS10', 'DGS5', 'FEDFUNDS', 'pred3_manual_dgs10_5', 'pred4_manual_dgs10_fedfunds']].sample(5)

In [ ]:
new_df[['cci','growth_30d','growth_snp500_30d','pred0_manual_cci','pred1_manual_prev_g1','pred2_manual_prev_g1_and_snp','is_positive_growth_30d_future']].sample(10)

In [ ]:
PREDICTIONS = [k for k in new_df.keys() if k.startswith('pred')]
PREDICTIONS

In [ ]:
p = PREDICTIONS[0]
part1 = p.split('_')[0] # first prefix before '_'
print(f'Full column name: {p}, only first part: {part1}')

In [ ]:
# One prediction: do we predict correctly?
new_df['is_correct_prediction'] = (new_df.pred0_manual_cci == new_df.is_positive_growth_30d_future)

In [ ]:
new_df[['cci','pred0_manual_cci','is_positive_growth_30d_future','is_correct_prediction']]

In [ ]:
# check "Precision" : the percentage of "correct" predictions , WHEN we predict "1" (POSITIVE future growth)
filter = (new_df.split=='test') & (new_df.pred0_manual_cci==1)
new_df[filter].is_correct_prediction.value_counts()


In [ ]:
# %% of correct predictions : 54%
new_df[filter].is_correct_prediction.value_counts() / len(new_df[filter])

In [ ]:
# delete this column
del new_df["is_correct_prediction"]

In [ ]:
# generate columns is_correct_
for pred in PREDICTIONS:
  part1 = pred.split('_')[0] # first prefix before '_'
  new_df[f'is_correct_{part1}'] =  (new_df[pred] == new_df.is_positive_growth_30d_future).astype(int)

In [ ]:
# IS_CORRECT dataset
IS_CORRECT =  [k for k in new_df.keys() if k.startswith('is_correct_')]
IS_CORRECT

In [ ]:
# sample of a dataframe
new_df[PREDICTIONS+IS_CORRECT+['is_positive_growth_30d_future']].sample(5)

In [ ]:
len(new_df[new_df.split=='test'])

In [ ]:
# define "Precision" for ALL predictions on a Test dataset (~4 last years of trading)
for i,column in enumerate(IS_CORRECT):
  prediction_column = PREDICTIONS[i]
  is_correct_column = column
  filter = (new_df.split=='test') & (new_df[prediction_column]==1)
  print(f'Prediction column:{prediction_column} , is_correct_column: {is_correct_column}')
  print(new_df[filter][is_correct_column].value_counts())
  print(new_df[filter][is_correct_column].value_counts()/len(new_df[filter]))

  print('---------')

## ✅ SOLUCIÓN – Pregunta 2 (parte B): precisión de pred3 y pred4 en TEST
**Precisión** = de todas las veces que la regla dijo “sube”, ¿qué porcentaje realmente subió?
`precisión = aciertos (TP) / (aciertos TP + falsas alarmas FP)`.
Si una regla nunca dice “sube” en el período TEST, no se le puede calcular precisión (no apostó nunca).

In [ ]:
# ✅ SOLUCIÓN — Pregunta 2 (parte B)
test_q2 = new_df[new_df.split == 'test']
precisiones_q2 = {}
for col in ['pred3_manual_dgs10_5', 'pred4_manual_dgs10_fedfunds']:
    apuestas = test_q2[test_q2[col] == 1]                      # días donde la regla dijo "sube"
    if len(apuestas) == 0:
        print(f'{col}: no hace ninguna predicción positiva en TEST')
        continue
    tp = (apuestas['is_positive_growth_30d_future'] == 1).sum()   # aciertos
    fp = len(apuestas) - tp                                       # falsas alarmas
    precisiones_q2[col] = tp / (tp + fp)
    print(f'{col}: predicciones=1 -> {len(apuestas)}, TP={tp}, FP={fp}, precisión={precisiones_q2[col]:.3f}')

respuesta_q2 = round(max(precisiones_q2.values()), 3)
print('RESPUESTA Q2:', respuesta_q2)

## 1.3) [Code Snippet 4 - Advanced] Statistical prediction : ARIMA models with 3 parameters (p,q,r)
* Finding best ARIMA params: not used in 2025

In [ ]:

import pandas as pd

# Set 'Date' as the index
df_arima = df_amzn.copy()

df_arima.set_index('Date', inplace=True)
df_arima.head()

# use ONLY time-series values and no other features
train_validation = df_arima[df_arima.split.isin(['train','validation'])]['Close_x']
test = df_arima[df_arima.split.isin(['test'])]['Close_x']

# this is needed to make a decision about investing for 5 days
test_is_positive_future_30d = df_arima[df_arima.split.isin(['test'])]['is_positive_growth_30d_future']


In [ ]:
# !pip install pmdarima
# from pmdarima import auto_arima

In [ ]:
# Hyper parameters optimization
# # ~ several minutes to run
# # https://miqbalrp.medium.com/exploring-autoarima-in-python-for-multiple-time-series-forecasting-2f3004ba5a49
# # https://alkaline-ml.com/pmdarima/modules/generated/pmdarima.arima.auto_arima.html
# # Auto ARIMA model fitting: iterating across params
# best_arima_model = auto_arima(train_validation,
#                    seasonal=False,
#                    trace=True,
#                    start_p=0, start_q=0, # minimum p and q
#                    max_p=12, max_q=12,   # maximum p and q
#                    D=None,               # let model determine 'D'
#                    )

In [ ]:
# best model specification trained on train_validation
# # (p=6, q=2, r=2)
# best_arima_model.get_params()['order']

# model parameters ("coef" for auto-regression (AR) and moving-average (MA))
# best_arima_model.summary()

In [ ]:
# need to do 918 predictions for 1 stock
len(test)

In [ ]:
from matplotlib import pyplot
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
from math import sqrt

In [ ]:
# [last day] this was used to train the model and obtain the "best" params
train_validation[-1:]

In [ ]:
# # history as Series and the "last value" (current last known data)
history = [x for x in train_validation]
history[-1]

In [ ]:
# [last day] now need to predict for each value of test, using all available history
test[-1:]

In [ ]:
# slow to predict, can't quickly predict for ALL 33 tickers * 918 observationa (~30k predictions). Let's have only 1 ticker and these periods to predict:
# change this to 10 (1 min) or 50 (5min) to run faster

PERIODS_TO_PREDICT = 10  # ~2 minute to predict
# PERIODS_TO_PREDICT = 500  #VERY SLOW: used for slides

In [ ]:
# staring to predict at day1 of test dataframe (we SEE the adj_close price and predict 5 periods ahead)
df_arima[df_arima.split=='test'].head(5)

In [ ]:
# 2026: static params, instead of autotuned

# p,q,r = best_arima_model.get_params()['order']
p,q,r = 6,2,2

In [ ]:
%%time
import numpy as np

# rolling forecast for ARIMA(6,2,2) model
# https://machinelearningmastery.com/arima-for-time-series-forecasting-with-python/


# start from all previous history of train_validation
history = [x for x in train_validation]
predictions = []
decisions = []
is_correct_decisions = []

tp=0 # true positive
fp=0 # false positive

# walk-forward validation for PERIODS_TO_PREDICT
for t in range(PERIODS_TO_PREDICT):
  current_adj_close = test.iloc[t] # current adj_close price when we do the prediction
  current_is_future_growth_30d = test_is_positive_future_30d.iloc[t]

  model = ARIMA(history, order = (p,q,r)) # substitute best parameters AND RETRAIN the model (very expensive)
  model_fit = model.fit()
  output = model_fit.forecast(steps=30) # forecast for 30 periods ahead

  yhat = output[29] # prediction 30 periods ahead

  if yhat > current_adj_close:
    prediction = 1
  else:
    prediction = 0

  is_correct = (prediction==current_is_future_growth_30d).astype(int)

  if prediction == 1:
    if is_correct ==1:
      tp+=1
    else:
      fp+=1

  predictions.append(yhat)

  decisions.append(prediction)
  is_correct_decisions.append(is_correct)
  # Corrected: Add the current actual observation to history for the next iteration
  history.append(current_adj_close)
  # Ensuring enough test data exists to get the actual 30-day ahead value for printing
  actual_30d_ahead = test.iloc[t + 29] if (t + 29) < len(test) else np.nan
  print(f'step {t}, current_price = {np.round(current_adj_close,1)}, predicted_30d_ahead={np.round(yhat,1)}, actual_30d_ahead={np.round(actual_30d_ahead,1)}, decision = {prediction}, is_correct = {is_correct} ')

In [ ]:
tp

In [ ]:
fp

In [ ]:
precision = tp / (tp+fp)
precision

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,8))

# Plot train+validation history
plt.plot(train_validation.index, train_validation, color='green', label = 'Train+Validation Stock Price')

# Ensure we have enough data in 'test' for the 30-day ahead forecasts
# We are forecasting 30 days ahead. So the predictions[t] corresponds to test.iloc[t+29]
# The actual prices to compare against are test.iloc[29 : PERIODS_TO_PREDICT + 29]
start_idx_for_plot = 29
end_idx_for_plot = PERIODS_TO_PREDICT + 29

if end_idx_for_plot <= len(test):
    actual_forecast_dates = test.index[start_idx_for_plot : end_idx_for_plot]
    actual_forecast_values = test.iloc[start_idx_for_plot : end_idx_for_plot]

    # Plot actual test prices for the forecast horizon (30 days ahead)
    plt.plot(actual_forecast_dates, actual_forecast_values, color = 'red', label = 'Real Stock Price (30 days ahead)')

    # Plot predicted prices (30 days ahead)
    plt.plot(actual_forecast_dates, predictions, color = 'blue', label = 'Predicted Stock Price (30 days ahead)')
else:
    print(f"Not enough data in 'test' for {PERIODS_TO_PREDICT} 30-day ahead predictions to align for plotting.")
    print(f"'test' length: {len(test)}, required length for plotting {end_idx_for_plot}")

plt.title('AMZN Stock Price Prediction (ARIMA)')
plt.xlabel('Date')
plt.ylabel('AMZN Stock Price')
plt.legend()
plt.grid(True)
plt.show()

## 1.4) [Code Snippet 5] Binary Decision Tree

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

### 1.4.1) Define dataframes AND perform data cleaning
* define X_train (dataframe), X_test (dataframe), y_train (series), y_test (series)
* replace +-inf. with 0
* fill NaNs with 0 (you can drop it too, but will loose a lot of data in our case
* remove 1-2% outliers (in each dimension, or only in variable to_predict :: we won't use it for a Decision Tree

In [ ]:
# Decision Tree doesn't like too large and inf. values
import numpy as np

def remove_infinite_values(X):
    """
    Remove infinite values from the input array.

    Parameters:
    - X: Input array (NumPy array or array-like)

    Returns:
    - Array with infinite values removed
    """
    return X[np.isfinite(X).all(axis=1)]

# Example usage:
# Assuming X is your input data
# filtered_X = remove_infinite_values(X)

In [ ]:
# look carefully for 'count' to be close to total values (or you need to replace NaNs/remove NaNs), and min/max doesn't equal to -+inf.
#  it will give you an idea to dig deeper into some features to understand the 'nature' of a problem
pd.set_option('display.max_rows', None)

new_df[NUMERICAL+DUMMIES].describe().T

In [ ]:
# Split the data into training and testing sets based on the split date
features_list = NUMERICAL+DUMMIES
to_predict = 'is_positive_growth_30d_future'

train_df = new_df[new_df.split.isin(['train','validation'])].copy(deep=True)
test_df = new_df[new_df.split.isin(['test'])].copy(deep=True)

# ONLY numerical Separate features and target variable for training and testing sets
# need Date and Ticker later when merging predictions to the dataset
X_train = train_df[features_list+[to_predict,'Date','Ticker']]
X_test = test_df[features_list+[to_predict,'Date','Ticker']]

print(f'length: X_train {X_train.shape},  X_test {X_test.shape}')


In [ ]:
# Can't have +-inf values . E.g. ln(volume)=-inf when volume==0 => substitute with 0

# Disable SettingWithCopyWarning
pd.options.mode.chained_assignment = None  # default='warn'

X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Need to fill NaNs somehow
X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

print(f'length: X_train_imputed {X_train.shape},  X_test_imputed {X_test.shape}')

In [ ]:
# you may want to remove 1-2% outliers based on percentile ==> not used here in Decision Trees
def remove_outliers_percentile(X, lower_percentile=1, upper_percentile=99):
    """
    Remove outliers from the input array based on percentiles.

    Parameters:
    - X: Input array (NumPy array or array-like)
    - lower_percentile: Lower percentile threshold (float, default=1)
    - upper_percentile: Upper percentile threshold (float, default=99)

    Returns:
    - Array with outliers removed
    """
    lower_bound = np.percentile(X, lower_percentile, axis=0)
    upper_bound = np.percentile(X, upper_percentile, axis=0)
    mask = np.logical_and(np.all(X >= lower_bound, axis=1), np.all(X <= upper_bound, axis=1))
    return X[mask]

# Example usage:
# Assuming X is your input data
# filtered_X = remove_outliers_percentile(X, lower_percentile=1, upper_percentile=99)

In [ ]:
X_train_imputed = X_train # we won't use outliers removal to save more data to train: remove_outliers_percentile(X_train)
X_test_imputed = X_test # we won't use outliers removal to save more data to test: remove_outliers_percentile(X_test)

In [ ]:
# same shape
print(f'length: X_train_imputed {X_train_imputed.shape},  X_test_imputed {X_test_imputed.shape}')

In [ ]:
y_train = X_train_imputed[to_predict]
y_test = X_test_imputed[to_predict]

# remove y_train, y_test from X_ dataframes
del X_train_imputed[to_predict]
del X_test_imputed[to_predict]

### 1.4.2 Estimation of a Decision Tree model

In [ ]:
# INPUTS:
# X_train_imputed : CLEAN dataFrame with only numerical features (train+validation periods)
# X_test_imputed : CLEAN dataFrame with only numerical features (test periods)

# y_train : true values for the train period
# y_test  : true values for the test period

In [ ]:
# estimation/fit function (using dataframe of features X and what to predict y) --> optimising total accuracy
# max_depth is hyperParameter
def fit_decision_tree(X, y, max_depth=20):
# Initialize the Decision Tree Classifier
  clf = DecisionTreeClassifier(max_depth=max_depth)

  # Fit the classifier to the training data
  clf.fit(X, y)
  return clf, X.columns

In [ ]:
%%time
# drop 2 columns before fitting the tree, but we need those columns later for joins
clf_20, train_columns = fit_decision_tree(X=X_train_imputed.drop(['Date','Ticker'],axis=1),
                           y=y_train,
                           max_depth=20)

In [ ]:
%%time
clf_10, train_columns = fit_decision_tree(X=X_train_imputed.drop(['Date','Ticker'],axis=1),
                           y=y_train,
                           max_depth=10)

In [ ]:
# TODO 3: TRAIN only on train dataset, experiment with trees with depth 1..20 --> find the best one on VALID dataset
#       for the "best" tree model: find precision on the TEST set

### 1.4.3 Inference for a Decision Tree

In [ ]:
def predict_decision_tree(clf:DecisionTreeClassifier, df_X:pd.DataFrame, y_true: pd.Series):
  # Predict the target variable on the test data
  y_pred = clf.predict(df_X)

  max_depth = clf.tree_.max_depth
  # Print the maximum depth
  print("Maximum depth of the decision tree:", max_depth)

  # Calculate the accuracy/precision of the model
  accuracy = accuracy_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred)
  print(f'Accuracy ={accuracy}, precision = {precision}')

  # resulting df
  result_df = pd.concat([df_X, y_true, pd.Series(y_pred, index=df_X.index, name='pred_')], axis=1)

  return result_df

In [ ]:
pred20 = predict_decision_tree(clf_20, X_test_imputed.drop(['Date','Ticker'],axis=1), y_test)

In [ ]:
# Predictions of a decision tree of depth "20"
pred20.pred_.value_counts()

In [ ]:
pred10 = predict_decision_tree(clf_10, X_test_imputed.drop(['Date','Ticker'],axis=1), y_test)

In [ ]:
pred10.tail()

In [ ]:
import numpy as np
import pandas as pd

# Use globally defined NUMERICAL, DUMMIES, and TO_PREDICT
# These should have been defined in earlier cells (e.g., nakB0XZA1uFq, sxLHnVE3RaaU, G7kZrwZiIpCN).
global_NUMERICAL = globals().get('NUMERICAL', [])
global_DUMMIES = globals().get('DUMMIES', [])
global_TO_PREDICT = globals().get('TO_PREDICT', [])

features_list_from_globals = global_NUMERICAL + global_DUMMIES
to_predict_col = global_TO_PREDICT[1] if len(global_TO_PREDICT) > 1 else 'is_positive_growth_30d_future' # Fallback

# Ensure new_df is available and properly structured
if 'new_df' not in globals() or globals()['new_df'].empty:
    raise RuntimeError("new_df is not defined or is empty. Please run all preceding data loading and preprocessing cells.")

test_df = new_df[new_df.split.isin(['test'])].copy(deep=True)

# Filter features_list_from_globals to include only columns actually present in test_df
valid_features = [col for col in features_list_from_globals if col in test_df.columns]

# Construct the final list of columns for X_test
X_test_columns_to_select = valid_features + ['Date', 'Ticker']
if to_predict_col in test_df.columns and to_predict_col not in X_test_columns_to_select:
    X_test_columns_to_select.append(to_predict_col)

# Re-create X_test with all necessary columns using the filtered list
X_test = test_df[X_test_columns_to_select].copy() # .copy() to ensure it's not a view

# Handle inf and NaN values (as done in cell 46GYMy7KRUB3)
pd.options.mode.chained_assignment = None # Disable SettingWithCopyWarning temporarily
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.fillna(0, inplace=True)
pd.options.mode.chained_assignment = 'warn' # Re-enable the warning

# Assign to X_test_imputed (as done in cell 8-NNrhAKWXr8)
X_test_imputed = X_test.copy() # Make a copy to prevent further SettingWithCopyWarning

# Delete to_predict column from X_test_imputed (as done in cell E6A_zhDLW0aC)
if to_predict_col in X_test_imputed.columns:
    del X_test_imputed[to_predict_col]

print(f'X_test_imputed has been recreated with shape: {X_test_imputed.shape}')

In [ ]:
X_test_imputed.join(pred10['pred_']).head()

In [ ]:
# Merge Decision Tree predictions with the main DataFrame
# Use 'Date' and 'Ticker' as merge keys as they are present in both `new_df` and `pred_df`s

# pred10 and pred20 dataframes were created using X_test_imputed.drop(['Date','Ticker'],axis=1)
# so they do NOT contain 'Date' and 'Ticker' directly.
# We need to re-add 'Date' and 'Ticker' from X_test_imputed based on their index.

# Get Date and Ticker from X_test_imputed, preserving the original index
date_ticker_info = X_test_imputed[['Date', 'Ticker']]

# Ensure pred10 and pred20 have 'Date' and 'Ticker' for merging
# Since pred10 and pred20 are created with df_X and y_true, they share the index of X_test_imputed
pred10_with_date_ticker = pred10.join(date_ticker_info)
pred20_with_date_ticker = pred20.join(date_ticker_info)

# Now, create pred10_df and pred20_df from these new DataFrames
pred10_df = pred10_with_date_ticker[['Date', 'Ticker']].copy()
pred10_df['pred_tree_clf10'] = pred10_with_date_ticker['pred_'].values

pred20_df = pred20_with_date_ticker[['Date', 'Ticker']].copy()
pred20_df['pred_tree_clf20'] = pred20_with_date_ticker['pred_'].values

# Make sure the 'split' column in new_df is used for filtering only the 'test' set for merging
# We need to make sure new_df is not modified before extracting the test subset
new_df_test_subset = new_df[new_df['split'] == 'test'].copy()

# Merge pred10_df
merged_df = pd.merge(new_df_test_subset, pred10_df[['Date', 'Ticker', 'pred_tree_clf10']],
                     on=['Date', 'Ticker'], how='left')

# Merge pred20_df
merged_df = pd.merge(merged_df, pred20_df[['Date', 'Ticker', 'pred_tree_clf20']],
                     on=['Date', 'Ticker'], how='left')

# Update the original new_df for the test split
# Ensure Date and Ticker are proper types for indexing/merging if needed
# Temporarily set index for efficient updating
# Use .copy() to avoid SettingWithCopyWarning
new_df_copy = new_df.copy()
new_df_indexed = new_df_copy.set_index(['Date', 'Ticker'], drop=False)
merged_df_indexed = merged_df.set_index(['Date', 'Ticker'], drop=False)

# Update only the 'test' split rows in new_df
# Use .loc with a list of columns for setting multiple columns
new_df_indexed.loc[merged_df_indexed.index, ['pred_tree_clf10', 'pred_tree_clf20']] = \
    merged_df_indexed[['pred_tree_clf10', 'pred_tree_clf20']]

# Since index keys were kept as columns (drop=False), we drop them from the index during reset to prevent duplication
new_df = new_df_indexed.reset_index(drop=True)

print("Decision tree predictions merged into new_df.")
print(new_df[['Date', 'Ticker', 'pred_tree_clf10', 'pred_tree_clf20']].tail())

In [ ]:
# Predictions of a decision tree of depth "10" : many more "positive" predictions
pred10.pred_.value_counts()

In [ ]:
import numpy as np
import pandas as pd

# Reconstruct X_test_imputed if it's not available in the global scope
if 'X_test_imputed' not in globals():
    print("X_test_imputed not found, reconstructing for robustness...")
    # Assuming NUMERICAL, DUMMIES, and TO_PREDICT are defined in previous cells
    features_list = globals().get('NUMERICAL', []) + globals().get('DUMMIES', [])
    to_predict_col = globals().get('TO_PREDICT', ['', 'is_positive_growth_30d_future'])[1]

    # Ensure new_df is available and properly structured
    if 'new_df' not in globals() or globals()['new_df'].empty:
        raise RuntimeError("new_df is not defined or is empty. Please run all preceding data loading and preprocessing cells.")

    temp_test_df = globals()['new_df'][globals()['new_df'].split.isin(['test'])].copy(deep=True)
    temp_X_test = temp_test_df[features_list + [to_predict_col, 'Date', 'Ticker']].copy()

    pd.options.mode.chained_assignment = None # Disable SettingWithCopyWarning
    temp_X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
    temp_X_test.fillna(0, inplace=True)
    pd.options.mode.chained_assignment = 'warn' # Re-enable

    X_test_imputed = temp_X_test.copy()
    if to_predict_col in X_test_imputed.columns:
        del X_test_imputed[to_predict_col]
    print(f"X_test_imputed reconstructed with shape: {X_test_imputed.shape}")
else:
    print("X_test_imputed is already available.")

# define a new DF with the SAME index (used for joins)
pred20_df = X_test_imputed[['Date', 'Ticker']].copy()
pred20_df['pred_tree_clf20'] = pred20['pred_'].values
pred20_df.head(1)

In [ ]:
import numpy as np
import pandas as pd

# Reconstruct X_test_imputed if it's not available in the global scope
# This provides redundancy for robustness if cells are run out of order, though _pgIZ95bmLD3 should define it.
if 'X_test_imputed' not in globals():
    print("X_test_imputed not found, reconstructing for robustness...")
    features_list = globals().get('NUMERICAL', []) + globals().get('DUMMIES', [])
    to_predict_col = globals().get('TO_PREDICT', ['', 'is_positive_growth_30d_future'])[1]

    if 'new_df' not in globals() or globals()['new_df'].empty:
        raise RuntimeError("new_df is not defined or is empty. Please run all preceding data loading and preprocessing cells.")

    temp_test_df = globals()['new_df'][globals()['new_df'].split.isin(['test'])].copy(deep=True)
    temp_X_test = temp_test_df[features_list + [to_predict_col, 'Date', 'Ticker']].copy()

    pd.options.mode.chained_assignment = None
    temp_X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
    temp_X_test.fillna(0, inplace=True)
    pd.options.mode.chained_assignment = 'warn'

    X_test_imputed = temp_X_test.copy()
    if to_predict_col in X_test_imputed.columns:
        del X_test_imputed[to_predict_col]
    print(f"X_test_imputed reconstructed with shape: {X_test_imputed.shape}")
else:
    print("X_test_imputed is already available.")

# define a new DF with the SAME index (used for joins)
pred10_df = X_test_imputed[['Date', 'Ticker']].copy()
pred10_df['pred_tree_clf10'] = pred10['pred_'].values
pred10_df.head(1)

### 1.4.4 Features Importance and Tree Visualisation of top levels (for clf10)

In [ ]:
# visualisation: decision tree for a few levels (max_depth variable)
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Assuming clf is your trained DecisionTreeClassifier
plt.figure(figsize=(20,10))  # Set the size of the figure
plot_tree(clf_10,
          filled=True,
          feature_names=train_columns,
          class_names=['Negative', 'Positive'],
          max_depth=2)
plt.show()

In [ ]:
# Feautures importance function to predict future returns (based on the classifier)
# get feature importance from 'clf' (classifier) and 'train_columns' (column names)

def get_importances(clf, train_columns):
  # Assuming clf is your trained DecisionTreeClassifier
  feature_importance = clf.feature_importances_

  # Assuming X_train is your training features
  feature_names = train_columns

  # Create a DataFrame to store feature importance
  feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importance})

  # Sort the DataFrame by importance in descending order
  feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

  # Print or display the feature importance DataFrame
  # print(feature_importance_df)
  return feature_importance_df

In [ ]:
get_importances(clf_10, train_columns).head(10)

In [ ]:
get_importances(clf_20, train_columns).head(10)

### 1.4.5 Merge with the original df for predictions (only when predicted on test dataset)

In [ ]:
# current predictions from MANUAL
PREDICTIONS

In [ ]:
new_df.head()

In [ ]:
# index in df is not unique
np.sort(new_df.groupby(new_df.index).split.count())

In [ ]:
# it's hard to join with pred10_df - as index is totally different
pred10_df.head()

In [ ]:
# TODO 4: JOIN predictions with the original dataframe (define a new column):
#  so, that there are columns pred_tree_clf10 AND pred_tree_clf20

---
# ✅ SOLUCIÓN – Preguntas 3, 4 y 5
Todo lo de arriba (salvo las celdas ✅) es código original del curso. Desde aquí hacia abajo es la solución.

## ✅ Pregunta 3: ¿cuántas veces acierta SOLO el árbol y fallan TODAS las reglas a mano?
**Analogía:** si un nuevo empleado solo acierta en los mismos casos que ya acertaban los empleados viejos, no aporta nada. Queremos contar los casos en que **únicamente** el modelo de Machine Learning (el árbol) tuvo razón.

**Pasos:**
1. **`X_all`:** tomamos TODO el dataset (train + validation + test) con las mismas columnas y la misma limpieza que se usó para entrenar (cambiar infinitos por vacío y los vacíos por 0). El árbol no acepta valores vacíos ni infinitos.
2. **Entrenar:** un árbol de decisión de profundidad 10 (`max_depth=10`, como máximo 10 preguntas seguidas del tipo “¿DGS10 ≤ 4?”) con `random_state=42` para que siempre salga el mismo resultado. Se entrena solo con train + validation (el pasado).
3. **Predecir** sobre todo `X_all` y guardar el resultado en `new_df['pred5_clf_10']`. Como tiene las mismas filas y en el mismo orden, se pega directo.
4. **Contar:** `only_pred5_is_correct` = 1 cuando el árbol acierta **y** pred0, pred1, pred2, pred3 y pred4 se equivocan. Sumamos esos 1 solo en TEST.

In [ ]:
# ✅ SOLUCIÓN — Pregunta 3, paso 1: X_all = todo el dataset, misma limpieza que X_train / X_test
X_all = new_df[features_list].copy()
X_all.replace([np.inf, -np.inf], np.nan, inplace=True)
X_all.fillna(0, inplace=True)
print('X_all:', X_all.shape, '| new_df:', new_df.shape)

In [ ]:
%%time
# ✅ SOLUCIÓN — Pregunta 3, paso 2: árbol de profundidad 10, entrenado en TRAIN + VALIDATION
X_tr = X_train_imputed.drop(['Date', 'Ticker'], axis=1)   # mismas columnas que X_all
clf_10_rs = DecisionTreeClassifier(max_depth=10, random_state=42)
clf_10_rs.fit(X_tr, y_train)

In [ ]:
# ✅ SOLUCIÓN — Pregunta 3, paso 3: predecir sobre TODO el dataset
new_df['pred5_clf_10'] = clf_10_rs.predict(X_all[X_tr.columns])
new_df['pred5_clf_10'].value_counts()

In [ ]:
# ✅ SOLUCIÓN — Pregunta 3, paso 4: aciertos únicos de pred5 en TEST
target = 'is_positive_growth_30d_future'
reglas_a_mano = ['pred0_manual_cci', 'pred1_manual_prev_g1', 'pred2_manual_prev_g1_and_snp',
                 'pred3_manual_dgs10_5', 'pred4_manual_dgs10_fedfunds']

# ¿acertó cada predicción? (1 = sí, 0 = no)
for col in reglas_a_mano + ['pred5_clf_10']:
    new_df[f"is_correct_{col.split('_')[0]}"] = (new_df[col] == new_df[target]).astype(int)

columnas_correctas_mano = [f"is_correct_{c.split('_')[0]}" for c in reglas_a_mano]   # is_correct_pred0 ... pred4

# 1 solo si pred5 acierta Y ninguna regla a mano acierta
new_df['only_pred5_is_correct'] = (
    (new_df['is_correct_pred5'] == 1) & (new_df[columnas_correctas_mano].sum(axis=1) == 0)
).astype(int)

respuesta_q3 = int(new_df[new_df.split == 'test']['only_pred5_is_correct'].sum())
print('RESPUESTA Q3:', respuesta_q3)

### (Opcional avanzado) Función general: ¿qué predicción acierta sola?
En vez de escribir la condición a mano para pred5, esta función sirve para cualquier columna: devuelve 1 si esa predicción acierta y todas las demás fallan.

In [ ]:
def es_unica_correcta(df_, columna_pred, todas_pred, target='is_positive_growth_30d_future'):
    correcta = df_[columna_pred] == df_[target]
    otras = [c for c in todas_pred if c != columna_pred]
    otras_correctas = sum((df_[c] == df_[target]).astype(int) for c in otras)
    return (correcta & (otras_correctas == 0)).astype(int)

todas = reglas_a_mano + ['pred5_clf_10']
test_q3 = new_df[new_df.split == 'test']
for c in todas:
    print(f'{c}: aciertos únicos en TEST = {es_unica_correcta(test_q3, c, todas).sum()}')

## ✅ Pregunta 4: ¿cuál es la mejor profundidad del árbol (1 a 12)?
**Analogía:** un árbol poco profundo es como un estudiante que solo aprendió reglas muy generales (se queda corto). Uno muy profundo es el que se memorizó el examen de práctica y falla en el examen real (*overfitting*, sobreajuste). Buscamos el punto medio.

**Pasos:**
1. Para cada profundidad de 1 a 12: entrenamos con train + validation y medimos la **precisión en TEST** (y también en el entrenamiento, para ver el sobreajuste).
2. La profundidad con mayor precisión en TEST es `best_max_depth`.
3. Reentrenamos con esa profundidad, predecimos todo el dataset y lo guardamos como `pred6_clf_best`.
4. Comparamos la precisión de pred0 … pred6.

In [ ]:
%%time
# ✅ SOLUCIÓN — Pregunta 4, paso 1: probar profundidades 1..12
from sklearn.metrics import precision_score

X_te = X_all[new_df.split == 'test'][X_tr.columns]
y_te = new_df.loc[new_df.split == 'test', 'is_positive_growth_30d_future'].fillna(0)

resultados = []
for depth in range(1, 13):
    modelo = DecisionTreeClassifier(max_depth=depth, random_state=42)
    modelo.fit(X_tr, y_train)
    prec_test = precision_score(y_te, modelo.predict(X_te), zero_division=0)
    prec_train = precision_score(y_train, modelo.predict(X_tr), zero_division=0)
    resultados.append({'max_depth': depth, 'precision_test': prec_test, 'precision_train': prec_train})
    print(f'max_depth={depth:2d} | precisión TRAIN+VAL={prec_train:.4f} | precisión TEST={prec_test:.4f}')

resultados_df = pd.DataFrame(resultados)
best_max_depth = int(resultados_df.loc[resultados_df.precision_test.idxmax(), 'max_depth'])
respuesta_q4 = best_max_depth
print('RESPUESTA Q4 (mejor max_depth):', best_max_depth)

In [ ]:
# (Opcional) precisión vs profundidad: TRAIN sigue subiendo, TEST se estanca o baja -> sobreajuste
fig = px.line(resultados_df, x='max_depth', y=['precision_train', 'precision_test'], markers=True,
              title='Precisión del árbol según su profundidad (max_depth)')
fig.show()

In [ ]:
# ✅ SOLUCIÓN — Pregunta 4, paso 2: árbol final con best_max_depth y comparación
from sklearn.tree import export_text

clf_best = DecisionTreeClassifier(max_depth=best_max_depth, random_state=42)
clf_best.fit(X_tr, y_train)
new_df['pred6_clf_best'] = clf_best.predict(X_all[X_tr.columns])

print(export_text(clf_best, feature_names=list(X_tr.columns), max_depth=3))

test_q4 = new_df[new_df.split == 'test']
comparacion = []
for col in reglas_a_mano + ['pred5_clf_10', 'pred6_clf_best']:
    apuestas = test_q4[test_q4[col] == 1]
    prec = (apuestas['is_positive_growth_30d_future'] == 1).mean() if len(apuestas) else np.nan
    comparacion.append({'prediccion': col, 'veces_que_dijo_sube': len(apuestas), 'precision_test': round(prec, 4)})
pd.DataFrame(comparacion)

## ✅ Pregunta 5 (exploratoria): ¿qué datos faltan?
Los árboles muestran que las variables **macro** (tasas de bonos `DGS10`/`DGS5`, tasa de la Fed, inflación) pesan mucho. Pero el dataset tiene acciones de **EE. UU., Europa e India** y casi toda la información macro es de EE. UU. Agregaría:

1. **Tasas e inflación locales:** tasa del Banco Central Europeo y del Banco de la Reserva de la India, e inflación de la eurozona y de la India. Fuente: FRED (`ECBDFR`, series de India) y las webs del BCE y del RBI.
2. **Tipos de cambio:** EUR/USD y USD/INR. Un inversionista en dólares gana o pierde también por la moneda. Fuente: `yfinance` (`EURUSD=X`, `INR=X`).
3. **Índice del miedo (VIX):** mide el nerviosismo del mercado; en pánico casi todo baja junto. Fuente: `yfinance` (`^VIX`).
4. **Precio del petróleo y del oro:** afectan a la inflación y a sectores enteros. Fuente: `yfinance` (`CL=F`, `GC=F`).
5. **Resultados trimestrales (earnings):** si la empresa ganó más o menos de lo esperado mueve el precio en los días siguientes. Fuente: `yfinance` (`Ticker.earnings_dates`).
6. **Índices locales:** cómo le va al mercado europeo (`^STOXX50E`) e indio (`^NSEI`), no solo al S&P 500. Fuente: `yfinance`.

In [ ]:
# 📝 Resumen de respuestas
pd.DataFrame({
    'Pregunta': ['Q1 – mayor |correlación| de month_wom',
                 'Q2 – precisión de la mejor regla nueva (pred3/pred4)',
                 'Q3 – aciertos únicos de pred5_clf_10 en TEST',
                 'Q4 – mejor max_depth (1–12)'],
    'Respuesta': [respuesta_q1, respuesta_q2, respuesta_q3, respuesta_q4]
})